# Debugging Next.js Middleware Cookie TypeScript Error

This notebook walks through fixing a Next.js middleware build failure caused by calling `req.cookies.set` with three arguments when the API expects either a `[key, value]` tuple or a `RequestCookie` object.

## 1. Analyze Build Error Output

Next.js build failed with a TypeScript error in `middleware.ts`:

- `Argument of type '[string, string, Partial<SerializeOptions>]' is not assignable to parameter of type '[key: string, value: string] | [options: RequestCookie]'.`
- This means the code is passing `name`, `value`, and `options` separately to `req.cookies.set`, while the current type definition only accepts a 2-tuple or a single `RequestCookie` object.

The problematic code is:

```ts
cookiesToSet.forEach(({ name, value, options }) => req.cookies.set(name, value, options));
```

## 2. Review Next.js Middleware Cookie API

The typed signature for `req.cookies.set` on `NextRequest` is:

- `req.cookies.set(name, value)` via tuple form
- `req.cookies.set(options)` via a single `RequestCookie` object

The current implementation incorrectly supplies three separate arguments.

We need to change it to either:

```ts
req.cookies.set(name, value);
```

or

```ts
req.cookies.set({ name, value, ...options });
```

The latter is the correct API when options are present.

## 3. Refactor `req.cookies.set` Call

Update `middleware.ts` to forward each cookie as a single `RequestCookie` object.

```ts
cookiesToSet.forEach(({ name, value, options }) => {
  req.cookies.set({ name, value, ...options });
});
```

This removes the incompatible third parameter and matches the updated Next.js type definition.

## 4. Add TypeScript Types for Middleware Parameters

Ensure the middleware callback is typed correctly.

```ts
import { type NextRequest } from 'next/server';

export async function middleware(req: NextRequest) {
  // ...
}
```

If `options` can be undefined, make sure the spread works safely:

```ts
req.cookies.set({ name, value, ...(options ?? {}) });
```

This guarantees TypeScript validates the request cookie object shape.

## 5. Rebuild and Verify the Fix

After updating `middleware.ts`, run:

```bash
npm run build
```

The build should now pass without the `req.cookies.set` type error.